# TimingGNN — 12-Feature Raw-Pipeline Training

**Model:** `gnn_model.py` — VN + sinusoidal level encoding + `at_init_mlp` + scheduled sampling + gradient detach  
**Data:** Raw DEF/SDF/SDC/Verilog pipeline via `parse_real_design` + `real_graph_to_tensors` (12 node features)  
**Split:** 15 train (14 DAC 2022 official + alu4bit) | 7 test (DAC 2022 fixed) | 5 calibration (held out — not used here)  
**Loss:** `2.0·AT + 1.0·SLEW + 0.3·net_delay + 0.3·cell_delay`  
**Selection metric:** mean-per-design AT R² (not pooled — avoids large-design domination)  

> **Leakage guard:** Hard assertions verify train ∩ test = ∅ and train ∩ calibration = ∅ before any data is loaded.

In [ ]:
import os, sys, copy, random, time, json, csv
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ── Add src/ to path so all project modules resolve ──────────────────────────
SRC_DIR  = r"C:\Krish\GCNN\src"
DATA_DIR = r"C:\Krish\GCNN\data"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from parse_real_design       import build_graph
from interactive_train_real  import real_graph_to_tensors, move_to_device, r2, DEVICE
from gnn_model               import ModelConfig, TimingGNN, NODE_IN_DIM
import report_writer

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"src/   : {SRC_DIR}")
print(f"data/  : {DATA_DIR}")

In [ ]:
# ── Design splits (FIXED — do not change without explicit discussion) ─────────
PAPER_TEST = [
    'aes192', 'jpeg_encoder', 'spm', 'synth_ram', 'usbf_device', 'xtea', 'y_huff'
]

CALIBRATION_DESIGNS = {
    'uni_counter4b', 'cla8b', 'gcd', 'seq_detector', 'priority_enc'
}

PAPER_TRAIN = [
    # DAC 2022 official 14 (derived via random.seed(8026728) + random.sample(available_data, 14))
    'BM64', 'aes128', 'aes256', 'aes_cipher', 'blabla',
    'cic_decimator', 'des', 'genericfir', 'picorv32a', 'salsa20',
    'usb', 'usb_cdc_core', 'wbqspiflash', 'zipdiv',
    # 1 extra (not in paper, not in calibration, not in test)
    'alu4bit',
]

# ── Hard leakage guard ────────────────────────────────────────────────────────
_test_set = set(PAPER_TEST)
for d in PAPER_TRAIN:
    assert d not in _test_set,          f"LEAKAGE: {d} is in PAPER_TEST"
    assert d not in CALIBRATION_DESIGNS, f"LEAKAGE: {d} is in CALIBRATION_DESIGNS"
for d in PAPER_TEST:
    assert d not in CALIBRATION_DESIGNS, f"LEAKAGE: {d} is in CALIBRATION_DESIGNS"

print(f"Split check OK:")
print(f"  {len(PAPER_TRAIN)} train  : {PAPER_TRAIN}")
print(f"  {len(PAPER_TEST)}  test   : {PAPER_TEST}")
print(f"  {len(CALIBRATION_DESIGNS)} calibration (held out, not used here): {sorted(CALIBRATION_DESIGNS)}")

In [ ]:
# ── Training hyper-parameters ─────────────────────────────────────────────────
HIDDEN_DIM   = 32
NET_LAYERS   = 3
LUT_K        = 7
EPOCHS       = 500
LR           = 2e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 25       # early-stop patience (epochs between evals, not raw epochs)
LOG_EVERY    = 10       # evaluate + print every N epochs
SEED         = 42

# Loss weights — AT is the metric that matters; delay heads must produce
# real-valued outputs (AT propagation consumes them) but shouldn't dominate.
W_AT   = 2.0
W_SLEW = 1.0
W_NET  = 0.3
W_CELL = 0.3

random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

cfg = ModelConfig(hidden_dim=HIDDEN_DIM, net_layers=NET_LAYERS, lut_k=LUT_K)
cfg.weight_decay        = WEIGHT_DECAY
cfg.early_stop_patience = PATIENCE

print(f"ModelConfig : {cfg.describe()}")
print(f"Epochs      : {EPOCHS}  |  LR: {LR}  |  patience: {PATIENCE}  |  log_every: {LOG_EVERY}")
print(f"Loss weights: AT={W_AT}  SLEW={W_SLEW}  net={W_NET}  cell={W_CELL}")

In [ ]:
# ── Confirm all required files exist on disk ──────────────────────────────────
def check_design(data_dir, name):
    d = os.path.join(data_dir, name)
    v   = os.path.exists(os.path.join(d, f"{name}.synthesis_preroute.v"))
    sdf = os.path.exists(os.path.join(d, f"{name}.sdf"))
    dfx = any(f.endswith(".def") for f in os.listdir(d)) if os.path.isdir(d) else False
    return v and sdf and dfx

missing_train = [d for d in PAPER_TRAIN if not check_design(DATA_DIR, d)]
missing_test  = [d for d in PAPER_TEST  if not check_design(DATA_DIR, d)]

if missing_train:
    print(f"WARNING — missing train designs (will be skipped): {missing_train}")
if missing_test:
    print(f"WARNING — missing test designs (will be skipped): {missing_test}")

train_designs = [d for d in PAPER_TRAIN if check_design(DATA_DIR, d)]
test_designs  = [d for d in PAPER_TEST  if check_design(DATA_DIR, d)]

print(f"\nReady to parse:")
print(f"  {len(train_designs)} train : {train_designs}")
print(f"  {len(test_designs)}  test  : {test_designs}")

In [ ]:
# ── Parse all designs ─────────────────────────────────────────────────────────
# This cell takes a few minutes — parsing verilog + DEF + SDF for 22 designs.
# Run once; re-run only if source files change.

print("Parsing train designs...")
train_raws = []
for name in train_designs:
    print(f"  {name}")
    raw = build_graph(os.path.join(DATA_DIR, name), name)
    train_raws.append((name, raw))

print("\nParsing test designs...")
test_raws = []
for name in test_designs:
    print(f"  {name}")
    raw = build_graph(os.path.join(DATA_DIR, name), name)
    test_raws.append((name, raw))

# Shared gate vocabulary across ALL designs (train + test)
all_gate_types = set()
for _, r in train_raws + test_raws:
    all_gate_types |= set(r["gate_of_out"].values())
shared_gate_types = sorted(all_gate_types)
if "UNKNOWN" not in shared_gate_types:
    shared_gate_types.append("UNKNOWN")

print(f"\nShared gate vocabulary: {len(shared_gate_types)} types")

In [ ]:
# ── Convert to tensors and move to device ─────────────────────────────────────
print("Converting to tensors...")
train_graphs = []
for name, raw in train_raws:
    g = move_to_device(real_graph_to_tensors(raw, shared_gate_types), DEVICE)
    train_graphs.append((name, g))
    print(f"  {name:15s} | {g['n_nodes']:6d} pins | {g['n_net_edges']:6d} net | {g['n_cell_edges']:6d} cell")

print()
test_graphs = []
for name, raw in test_raws:
    g = move_to_device(real_graph_to_tensors(raw, shared_gate_types), DEVICE)
    test_graphs.append((name, g))
    print(f"  {name:15s} | {g['n_nodes']:6d} pins | {g['n_net_edges']:6d} net | {g['n_cell_edges']:6d} cell")

# Verify feature consistency
in_dim   = train_graphs[0][1]["x"].shape[1]
edge_dim = train_graphs[0][1]["edge_geo"].shape[1]
for name, g in train_graphs + test_graphs:
    assert g["x"].shape[1] == in_dim, \
        f"{name}: node features {g['x'].shape[1]} != expected {in_dim}"
    assert g["edge_geo"].shape[1] == edge_dim, \
        f"{name}: edge features {g['edge_geo'].shape[1]} != expected {edge_dim}"

print(f"\nFeature dims: node={in_dim} (expected {NODE_IN_DIM})  edge={edge_dim}")
assert in_dim == NODE_IN_DIM, \
    f"Node dim mismatch: got {in_dim}, gnn_model expects {NODE_IN_DIM}"
print("Feature dim check PASSED")

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
model = TimingGNN(
    cfg,
    n_gates  = len(shared_gate_types),
    in_dim   = in_dim,
    edge_dim = edge_dim,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"TimingGNN ready")
print(f"  Trainable parameters : {n_params:,}")
print(f"  Gate types           : {len(shared_gate_types)}")
print(f"  Node features (in_dim): {in_dim}")
print(f"  Edge features         : {edge_dim}")
print(f"  Device                : {DEVICE}")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
# Matches train_paper_split.py exactly:
#   - Scheduled sampling: linear ramp teacher-forcing → self-prediction
#   - Per-graph coin flip (not per-batch average)
#   - Gradient clipping max_norm=1.0
#   - ReduceLROnPlateau on mean-per-design AT R²
#   - Early stop on mean-per-design AT R² with PATIENCE
#   - Best model weights restored at end

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode="max", factor=0.5, patience=max(3, EPOCHS // 20))
mse = nn.MSELoss()

def mean_per_design_r2(graphs):
    """Mean AT R² across test designs, weighting each design equally."""
    vals = []
    for _, g in graphs:
        out  = model(g)
        ep   = g["endpoints"]
        val  = r2(out["AT"][ep], g["AT_true"][ep])
        if val == val:  # skip NaN
            vals.append(val)
    return float(np.mean(vals)) if vals else float("nan")

best_r2     = -float("inf")
best_state  = None
best_epoch  = 0
patience_ctr = 0
epoch_log   = []   # (epoch, loss, mean_r2, pooled_r2, lr)

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    graphs_shuffled = list(train_graphs)
    random.shuffle(graphs_shuffled)
    total_loss = 0.0

    # Scheduled sampling: epoch 1 = full teacher forcing, final epoch = full self-predict
    p_gt = 1.0 - (epoch - 1) / max(1, EPOCHS - 1)

    for _, g in graphs_shuffled:
        use_gt = random.random() < p_gt
        out = model(g, groundtruth=use_gt)

        loss = (W_AT   * mse(out["AT"],         g["AT_true"])
              + W_SLEW * mse(out["SLEW"],        g["SLEW_true"])
              + W_NET  * mse(out["net_delay"],   g["nd_true"])
              + W_CELL * mse(out["cell_delay"],  g["cd_true"]))

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        total_loss += loss.item()

    avg_loss = total_loss / max(1, len(train_graphs))

    if epoch % LOG_EVERY == 0 or epoch == 1 or epoch == EPOCHS:
        model.eval()
        with torch.no_grad():
            te_mean = mean_per_design_r2(test_graphs)
            # pooled R² (for logging only — NOT the selection metric)
            all_pred, all_true = [], []
            for _, g in test_graphs:
                ep = g["endpoints"]
                all_pred.append(model(g)["AT"][ep])
                all_true.append(g["AT_true"][ep])
            te_pooled = r2(torch.cat(all_pred), torch.cat(all_true))

        scheduler.step(te_mean)
        lr_now = opt.param_groups[0]["lr"]

        if te_mean > best_r2 + 1e-4:
            best_r2    = te_mean
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
            flag = " ★"
        else:
            patience_ctr += 1
            flag = ""

        elapsed = time.time() - t0
        print(f"  ep {epoch:4d} | loss {avg_loss:7.4f} | mean-R2 {te_mean:.4f} | "
              f"pooled-R2 {te_pooled:.4f} | best {best_r2:.4f}@{best_epoch} | "
              f"lr {lr_now:.1e} | pat {patience_ctr}/{PATIENCE} | {elapsed:.0f}s{flag}")
        epoch_log.append((epoch, avg_loss, te_mean, te_pooled, lr_now))

        if patience_ctr >= PATIENCE:
            print(f"\n  [early stop] epoch {epoch} — restoring best from epoch {best_epoch}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nTraining complete. Best mean-per-design AT R² = {best_r2:.4f} @ epoch {best_epoch}")
print(f"Total time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Per-design evaluation (best model weights loaded) ─────────────────────────
model.eval()
per_design_results = {}

print(f"{'design':15s} | {'AT R²':7s} | {'slack R²':8s} | {'SLEW R²':7s} | {'net_delay R²':12s} | {'cell_delay R²':13s}")
print("-" * 80)

with torch.no_grad():
    for name, g in test_graphs:
        out = model(g)
        ep  = g["endpoints"]

        at_r2   = r2(out["AT"][ep],    g["AT_true"][ep])
        slew_r2 = r2(out["SLEW"][ep],  g["SLEW_true"][ep])
        nd_r2   = r2(out["net_delay"],  g["nd_true"]) if g["nd_true"].numel() > 0 else float("nan")
        cd_r2   = r2(out["cell_delay"], g["cd_true"]) if g["cd_true"].numel() > 0 else float("nan")

        # Slack: use real RAT if available, else relative proxy
        slack_type = g.get("slack_type", "proxy")
        if slack_type in ("real_rat", "real"):
            rat        = g["slack_true"][ep] + g["AT_true"][ep]
            slack_pred = rat - out["AT"][ep]
            slack_true = g["slack_true"][ep]
        else:
            slack_pred = out["AT"][ep].max() - out["AT"][ep]
            slack_true = g["slack_true"][ep]
        slack_r2 = r2(slack_pred, slack_true)

        per_design_results[name] = {
            "AT_R2": at_r2, "slack_R2": slack_r2,
            "SLEW_R2": slew_r2, "net_delay_R2": nd_r2, "cell_delay_R2": cd_r2,
        }
        print(f"  {name:13s} | {at_r2:7.4f} | {slack_r2:8.4f} | {slew_r2:7.4f} | {nd_r2:12.4f} | {cd_r2:13.4f}")

at_vals = [v["AT_R2"] for v in per_design_results.values() if v["AT_R2"] == v["AT_R2"]]
print("-" * 80)
print(f"  {'MEAN':13s} | {np.mean(at_vals):7.4f}")
print(f"\nMean-per-design AT R² = {np.mean(at_vals):.4f}  (best epoch {best_epoch})")

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
# Creates: results/<timestamp>_train15_test7/
#   config.json, epoch_log.csv, predictions/<design>.csv
#   model_checkpoint.pt, source_snapshot/, report.md

RESULTS_DIR = os.path.join(SRC_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ts       = time.strftime("%Y%m%d_%H%M%S")
run_name = f"{ts}_train{len(train_designs)}_test{len(test_designs)}_12feat"
run_dir  = os.path.join(RESULTS_DIR, run_name)
os.makedirs(run_dir, exist_ok=True)

# config.json
config = {
    "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_config":    cfg.describe(),
    "loss_weights":    {"AT": W_AT, "SLEW": W_SLEW, "net": W_NET, "cell": W_CELL},
    "weight_decay":    WEIGHT_DECAY,
    "early_stop_patience": PATIENCE,
    "epochs_run":      epoch_log[-1][0] if epoch_log else 0,
    "best_epoch":      best_epoch,
    "best_mean_r2":    best_r2,
    "train_designs":   train_designs,
    "test_designs":    test_designs,
    "calibration_designs": sorted(CALIBRATION_DESIGNS),
    "n_params":        n_params,
    "per_design_final": per_design_results,
}
with open(os.path.join(run_dir, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

# epoch_log.csv
with open(os.path.join(run_dir, "epoch_log.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["epoch", "loss", "mean_per_design_AT_R2", "pooled_AT_R2", "lr"])
    for row in epoch_log:
        w.writerow(row)

# predictions/<design>.csv
pred_dir = os.path.join(run_dir, "predictions")
os.makedirs(pred_dir, exist_ok=True)
with torch.no_grad():
    for name, g in test_graphs:
        out = model(g)
        ep  = g["endpoints"]
        if g.get("slack_type") in ("real_rat", "real"):
            rat        = g["slack_true"][ep] + g["AT_true"][ep]
            slack_pred = (rat - out["AT"][ep]).cpu().tolist()
        else:
            slack_pred = (out["AT"][ep].max() - out["AT"][ep]).cpu().tolist()
        path = os.path.join(pred_dir, f"{name}.csv")
        with open(path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["design", "split", "pin_idx", "AT_true", "AT_pred", "slack_true", "slack_pred"])
            for pid, at_t, at_p, sl_t, sl_p in zip(
                    ep.cpu().tolist(),
                    g["AT_true"][ep].cpu().tolist(),
                    out["AT"][ep].cpu().tolist(),
                    g["slack_true"][ep].cpu().tolist(),
                    slack_pred):
                w.writerow([name, "test", pid, at_t, at_p, sl_t, sl_p])

# model checkpoint + source snapshot + report.md via report_writer
report_writer.save_full_run(
    run_dir, cfg,
    train_designs, test_designs,
    epoch_log,
    per_design_results,
    n_params, best_epoch, model,
    in_dim, edge_dim,
    len(shared_gate_types),
    gate_types=shared_gate_types,
)

print(f"Results saved to: {run_dir}")
print(f"  config.json, epoch_log.csv, predictions/{len(test_designs)} CSVs")
print(f"  model_checkpoint.pt, source_snapshot/, report.md")

In [ ]:
# ── Training curve ────────────────────────────────────────────────────────────
epochs_logged = [r[0] for r in epoch_log]
mean_r2s      = [r[2] for r in epoch_log]
pooled_r2s    = [r[3] for r in epoch_log]
losses        = [r[1] for r in epoch_log]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_logged, mean_r2s,   label="mean-per-design", lw=2)
axes[0].plot(epochs_logged, pooled_r2s, label="pooled",          lw=1, alpha=0.6)
axes[0].axvline(best_epoch, color="red", ls="--", alpha=0.5, label=f"best epoch {best_epoch}")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("AT R²")
axes[0].set_title("Test AT R² over training")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(epochs_logged, losses)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss (log)")
axes[1].set_title("Training loss"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(run_dir, "training_curve.png"), dpi=120)
plt.show()

In [ ]:
# ── Per-design AT scatter plots ───────────────────────────────────────────────
n = len(test_graphs)
cols = min(4, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.array(axes).flatten()

model.eval()
with torch.no_grad():
    for idx, (name, g) in enumerate(test_graphs):
        out = model(g)
        ep  = g["endpoints"]
        true_at = g["AT_true"][ep].cpu().numpy()
        pred_at = out["AT"][ep].cpu().numpy()
        at_r2_val = per_design_results[name]["AT_R2"]

        ax = axes[idx]
        ax.scatter(true_at, pred_at, s=2, alpha=0.3)
        lo = min(true_at.min(), pred_at.min())
        hi = max(true_at.max(), pred_at.max())
        ax.plot([lo, hi], [lo, hi], "r--", lw=1)
        ax.set_xlabel("AT true (ns)"); ax.set_ylabel("AT pred (ns)")
        ax.set_title(f"{name}\nR²={at_r2_val:.4f}")
        ax.grid(alpha=0.2)

for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f"AT scatter — mean-per-design R² = {np.mean(at_vals):.4f}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(run_dir, "at_scatter.png"), dpi=120, bbox_inches="tight")
plt.show()